In [0]:
df = spark.table("workspace.default.movies")

In [0]:
display(df.limit(5))
display(df.printSchema())
display(df.describe())

In [0]:
display(df.summary())

# Column Filtering

In [0]:
df_trimmed = df.select("title", "budget", "genres", "popularity", "release_date", "revenue", "runtime", "status", "title", "vote_average", "vote_count")
display(df_trimmed.limit(5))
display(df_trimmed.count())


In [0]:
from pyspark.sql.functions import col, to_date, format_number, year, rank

from pyspark.sql.window import Window

df_filtered = df_trimmed.withColumn("release_date", to_date(col("release_date"), "yyyy-MM-dd")) \
    .filter(
        (year(col("release_date")) > 2000) &
        (col("revenue") > 800000000) &
        ((col("revenue") - col("budget")) > 100000000)
    )

df_filtered = df_filtered.withColumn("profit", col("revenue") - col("budget"))

window_spec = Window.orderBy(col("profit").desc())

df_filtered = df_filtered.withColumn("rank", rank().over(window_spec)) \
    .orderBy(col("profit").desc()) \
    .limit(10) \
    .withColumn("revenue", format_number(col("revenue"), 0)) \
    .withColumn("budget", format_number(col("budget"), 0)) \
    .withColumn("profit", format_number(col("profit"), 0))

display(df_filtered)

#Performance Results
The explode() approach is faster - 0.679s vs 0.774s (about 14% faster)

# Key Differences
##Approach 1 (explode) - Better Performance
* Simpler execution: Scan → Explode → Filter → Shuffle → Aggregate
* Data flow: Creates 3 rows per movie (~14,400 rows), then filters with one Contains() check per row
* Resource usage: Single filter operation after explosion
* Plan complexity: Fewer operations in the execution plan
## Approach 2 (stack) - More Complex
* Filter pushdown: Applies 3 Contains() checks during table scan (good for filtering early)
* Extra computation: Evaluates 3 IF expressions per row before stack
* Data flow: Filters first, then creates 3 IF columns, then unpivots with stack, then filters NULLs
* Plan complexity: More operations (3 IFs + stack + NULL filter)
## Why explode() wins here:
* Dataset size: With only ~4,800 movies, exploding to ~14,400 rows isn't expensive
* Simpler logic: One filter operation vs three IF evaluations + NULL filtering
* Cleaner code: More readable and maintainable
* Photon optimization: Both use Photon acceleration, but the simpler plan executes faster
## When stack() might be better:
* Large datasets: If you had millions of rows, filter pushdown could reduce I/O significantly
* Selective filters: If only a tiny fraction of rows match, stack's early filtering would help more

Recommendation: Use the explode() approach - it's simpler, faster, and easier to understand for this use case.

In [0]:
from pyspark.sql.functions import explode, collect_list, col, array, lit

matrix_actors = ["Keanu Reeves", "Laurence Fishburne", "Carrie-Anne Moss"]

# Create a row for each movie-actor combination
df_cast_exploded = df.select("title", "cast", explode(array([lit(actor) for actor in matrix_actors])).alias("actor"))

# Filter where the actor name appears in the cast string, then aggregate
df_cast = df_cast_exploded.filter(col("cast").contains(col("actor"))) \
    .groupBy("actor") \
    .agg(collect_list("title").alias("movies"))

display(df_cast)

# Summary
The first query is fully local (no shuffle), while the second introduces a shuffle due to groupBy(actor), clearly visible via PhotonShuffleExchange.

## Flow
Stage 1:
  Scan → explode → filter

↓ SHUFFLE (by actor)

Stage 2:
  groupBy → collect_list

Performance implications

## Query 1:
Very fast

* No network
* No repartition
* Fully columnar (Photon)

## Query 2:
Expensive step "Shuffle by actor"

### Potential issues:
* Data skew (popular actors)
* Memory pressure
* Disk spill

In [0]:
df_cast_exploded.explain(True)
df_cast.explain(True)

In [0]:
from pyspark.sql.functions import col, collect_list, array, lit, expr

matrix_actors = ["Keanu Reeves", "Laurence Fishburne", "Carrie-Anne Moss"]

# Filter rows where cast contains any of the matrix_actors
df_cast = df.select("title", "cast") \
    .filter(
        expr(
            " OR ".join([f"cast LIKE '%{actor}%'" for actor in matrix_actors])
        )
    )

# For each actor, collect movies where their name appears in cast
df_actor_movies = df_cast.select(
    *[
        expr(f"IF(cast LIKE '%{actor}%', '{actor}', NULL)").alias(f"actor_{i}")
        for i, actor in enumerate(matrix_actors)
    ],
    "title"
)

from pyspark.sql.functions import collect_list

df_actor_matrix_movies = df_actor_movies.select(
    expr("stack(3, " +
         ", ".join([f"'{actor}', actor_{i}" for i, actor in enumerate(matrix_actors)]) +
         ") as (actor, actor_name)"),
    "title"
).filter(col("actor_name").isNotNull()) \
 .groupBy("actor") \
 .agg(collect_list("title").alias("movies"))

display(df_actor_matrix_movies)

# Summary
This version is more optimized than the previous one because it reduces data before expansion, but it still cannot avoid shuffle due to the GROUP BY actor, which remains the dominant cost.
## Execution flow (this plan)
Stage 1 (NO shuffle):
  Scan
  → Filter (Contains)
  → IF conditions
  → stack (row expansion)

↓ SHUFFLE (by actor)

Stage 2:
  groupBy(actor)
  → collect_list

## Improvements:
* Filters early 
* Reduces dataset before expansion 
* More efficient scan path 

**GOOD optimization:**
* Filter pushed down to scan
From plan:
* DataFilters: Contains(cast, ...)

**Means:**
* Less data read from storage
* Less memory usage

But it still:
* String scan
* No index
* Full column scan

## Subtle performance gains
Compared to previous version, this one:
* Reads less data
* Processes fewer rows pre-shuffle
* Reduces shuffle volume (important!)

In [0]:
df_cast.explain(True)
df_actor_movies.explain(True)
df_actor_matrix_movies.explain(True)

In [0]:
display(df_cast)

# Optimized version (no LIKE, structured, map-side friendly)
Because our cast field is unstructured text, we cannot eliminate string scanning—but we can simplify the plan by exploding the actor list instead of duplicating LIKE logic, while the real performance fix lies in restructuring the data model.
## Key idea:
* Less repetitive
* Slightly more efficient
* More maintainable

## Why this is better
### Previus Problems:
* Repeats logic N times
* Hard to scale (more actors = more columns)
* More CPU overhead

### New version:
explode actors → filter → groupBy

Benefits:
* Scales linearly with actor list
* Cleaner execution plan
* Fewer expressions

### Proper schema should be:
> cast: ARRAY<STRING>

or even better:
> movie_actor table:
(movie_id, actor)

Then our query becomes:
> df.filter(col("actor").isin(matrix_actors)) \
  .groupBy("actor") \
  .agg(collect_list("title"))

This:
* Avoids string scan
* Uses partitioning
* Minimizes shuffle

# Conclusion:
The previous version is worse because it does more work per row and repeats the same expensive string scans multiple times, while the second version does one pass with a simpler, linear plan in Apache Spark on Databricks.

In [0]:
from pyspark.sql.functions import col, collect_list, explode, array, lit

matrix_actors = ["Keanu Reeves", "Laurence Fishburne", "Carrie-Anne Moss"]

# 1. Create array of actors (small broadcast-like structure)
actors_df = df.select(
    "title",
    "cast",
    explode(array(*[lit(actor) for actor in matrix_actors])).alias("actor")
)

# 2. Filter using contains (still string-based, but cleaner)
df_filtered = actors_df.filter(
    col("cast").contains(col("actor"))
)

# 3. Aggregate
df_actor_matrix_movies = df_filtered.groupBy("actor") \
    .agg(collect_list("title").alias("movies"))

display(df_actor_matrix_movies)

# Make it REALLY efficient
The best approach is to query directly from a normalized movie_actor table using actor as a first-class column, optionally partitioned or pre-aggregated—this removes string processing entirely and reduces the problem to a simple filter + (optional) aggregation with minimal or no shuffle.
## Option A — Partition by actor

When writing the table:

> movie_actor_df.write.partitionBy("actor").format("delta").save(...)

Then query:
>spark.read.table("movie_actor") \
    .filter(col("actor").isin(matrix_actors))

Spark will:
* Read ONLY relevant partitions
* Skip most data
## Option B — Z-ORDER (Databricks-specific)
> OPTIMIZE movie_actor ZORDER BY (actor)

Improves:
* Data skipping
* File pruning
## Option C — Pre-aggregation (if repeated queries)
If this query is frequent:

> movie_actor_df.groupBy("actor") \
    .agg(collect_list("title").alias("movies")) \
    .write.format("delta").save("actor_movies")

Then:
> df.filter(col("actor").isin(matrix_actors))

**NO** shuffle at query time

## Best-practice recommendation
* Normalize into movie_actor
* Partition or ZORDER by actor
* Pre-aggregate if query is frequent

In [0]:

root
 |-- budget: long (nullable = true)
 |-- genres: string (nullable = true)
 |-- homepage: string (nullable = true)
 |-- id: long (nullable = true)
 |-- keywords: string (nullable = true)
 |-- original_language: string (nullable = true)
 |-- original_title: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- popularity: double (nullable = true)
 |-- production_companies: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- revenue: long (nullable = true)
 |-- runtime: double (nullable = true)
 |-- spoken_languages: string (nullable = true)
 |-- status: string (nullable = true)
 |-- tagline: string (nullable = true)
 |-- title: string (nullable = true)
 |-- vote_average: double (nullable = true)
 |-- vote_count: long (nullable = true)
 |-- cast: string (nullable = true)
 |-- crew: string (nullable = true)
 |-- director: string (nullable = true)


In [0]:
df.createOrReplaceTempView("movies")

In [0]:
%sql
SELECT DISTINCT genres from movies;

# Repartition And Coalesce
## Repartition
Repartitioning should be driven by how your data will be grouped, joined, or written—when aligned correctly, it reduces shuffle cost and improves performance; when misused, it becomes one of the most expensive operations in Spark.

### Golden rules (remember these)
* Repartition only with a purpose
* Align with downstream keys
A* void multiple repartitions
* Watch skew
* Balance file size vs parallelism
* Prefer structure over brute-force repartition

## Coalesce
If you need reduce the numer of partitions. Coalesce never increase the number of partitions
But it can increase the number of files and increase the shuffle
For example collapse 100 partitions → 15 will do:
* WITHOUT rebalancing data
* partitions are merged arbitrarily

On the end it will be 15 partitions, but possible data skew
### When coalesce IS useful
* Reducing small files AFTER heavy processing
* Final output tuning
* When data is already well distributed

In the example below using coalesce(15) after repartitioning by genres and release_date reduces the number of partitions without reshuffling, which destroys the careful data distribution by those columns and can lead to skewed partitions and less efficient writes.

In [0]:
from pyspark.sql.functions import col

df_repartitioned = df.repartition(
    20,
    col("release_date"),
    col("genres")
)

df_repartitioned.coalesce(15)


### Alternative: partition when writing (better for storage)
* Organizes files physically
* Enables partition pruning
* Avoids unnecessary runtime repartition later

In [0]:
df.repartition(20, col("release_date"), col("genres")
).write \
  .mode("overwrite") \
  .parquet("/Volumes/workspace/default/raw_data/movies_repartitioned")

# Option 2 — Use Delta (recommended in Databricks)
Delta handles:
* File cleanup
* Transaction consistency


In [0]:
df.repartition(20, col("release_date"), col("genres")) \
  .write \
  .format("delta") \
  .mode("overwrite") \
  .save("/Volumes/workspace/default/raw_data/movies_repartitioned")

# External Data - From AWS S3

In [0]:
%sql
SELECT
 *
FROM 
  read_files('s3://mgm-company-stocks/',
  format => 'csv',
  header => true);

In [0]:
%sql
CREATE TABLE workspace.default.company_stoks
USING DELTA
LOCATION 's3://mgm-company-stocks/bronze/'
AS
SELECT
 *,
 current_timestamp() as _ingested_at,
 input_file_name() AS _source_file,
 uuid() as _bronze_id
FROM 
  read_files('s3://mgm-company-stocks/',
  format => 'csv',
  header => true, 
  inferSchema => true);

In [0]:
%sql
UPDATE workspace.default.company_stoks
SET Stock_Price = 70
WHERE Company = "Apple";

In [0]:
%sql
select * from workspace.default.company_stoks where Company = "Apple";

In [0]:
%sql
select * from workspace.default.company_stoks version as of 0 where Company = "Apple";

# Databricks best practice
## Use:
* Avro → ingestion
* Delta → storage + analytics

## Medallion architecture:
* Bronze  → Avro / raw files
* Silver  → Delta (cleaned)
* Gold    → Delta (aggregated)